In [ ]:
import torch
import pandas as pd
import numpy as np
import random
from pyfaidx import Fasta

import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

# from model import SeqNN
from model_v2_compatible import SeqNN

In [ ]:
# functions

def one_hot_encode_sequence(sequence_obj):
    # Convert pyfaidx.Sequence object to string
    sequence = str(sequence_obj).upper()
    
    # Define the mapping from bases to integers
    base_to_int = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    valid_bases = list(base_to_int.keys())

    # Step 1: Convert sequence to integer encoding with random base for 'N'
    encoded_indices = []
    for base in sequence:
        if base in base_to_int:
            encoded_indices.append(base_to_int[base])
        else:
            random_base = random.choice(valid_bases)
            encoded_indices.append(base_to_int[random_base])

    # Step 2: One-hot encode the sequence
    encoded_sequence = np.array(encoded_indices)
    one_hot_encoded = np.zeros((4, len(encoded_sequence)), dtype=np.float32)
    one_hot_encoded[encoded_sequence, np.arange(len(encoded_sequence))] = 1

    input_sequence = np.expand_dims(one_hot_encoded, axis=0)

    return input_sequence



def upper_triangular_to_vector_skip_two_diagonals(matrix):
    """
    Extracts the upper triangular part of a square matrix (excluding the first two diagonals) 
    and transforms it into a vector.
    
    Parameters:
        matrix (np.ndarray): A 2D numpy matrix of shape (512, 512).
        
    Returns:
        np.ndarray: A 1D array containing the upper triangular elements (excluding the first two diagonals).
    """
    if matrix.shape != (512, 512):
        raise ValueError("Input matrix must be of shape (512, 512).")
    
    # Extract the upper triangular part excluding the first two diagonals
    upper_triangular_vector = matrix[np.triu_indices(512, k=2)]
    
    return upper_triangular_vector


def fragment_indices_in_upper_triangular(matrix_size=512, fragment_mask=None):
    """
    Given a binary mask for a fragment in a (448, 448) matrix, find the corresponding indices 
    within the upper triangular output vector (excluding the first two diagonals).

    Parameters:
        matrix_size (int): The size of the square matrix (default: 448).
        fragment_mask (np.ndarray): A boolean mask of shape (448, 448) marking the fragment.

    Returns:
        np.ndarray: Indices in the upper triangular vector corresponding to the fragment.
    """
    if fragment_mask.shape != (matrix_size, matrix_size):
        raise ValueError("Fragment mask must be of shape (448, 448).")

    # Get the upper triangular indices skipping two diagonals
    row_indices, col_indices = np.triu_indices(matrix_size, k=2)
    
    # Identify which of these indices are in the fragment
    selected_indices = np.where(fragment_mask[row_indices, col_indices])[0]
    
    return selected_indices



def store_tower_output(ohe_sequence, model, path):
    x = model.conv_block_1(ohe_sequence)
    x = model.conv_tower(x)
    # save the tensor
    torch.save(x, path)
    torch.cuda.empty_cache()

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
model_idx = 1

In [ ]:
model = SeqNN()
# model.load_state_dict(torch.load(f"/scratch1/smaruj/Akita_pytorch_models/finetuned/mouse_models/Hsieh2019_mESC/models/Akita_v2_mouse_Hsieh2019_mESC_model{model_idx}_finetuned.pth", map_location=device))
model.load_state_dict(torch.load(f"/scratch1/smaruj/Akita_pytorch_models/finetuned/mouse_models/Bonev2017_ncx_NPC/models/Akita_v2_mouse_Bonev2017_ncx_NPC_model{model_idx}_finetuned.pth", map_location=device))
model.eval()

In [ ]:
FOLD = 0

In [ ]:
df = pd.read_csv(f"/scratch1/smaruj/genomic_flat_regions/flat_regions_chrom_states_tsv/fold{FOLD}_selected_genomic_windows_centered_chrom_states.tsv", sep="\t")

In [ ]:
fasta_file = "/project2/fudenber_735/genomes/mm10/mm10.fa"
genome = Fasta(fasta_file)

In [ ]:
# c = -0.2 # how strong target boundary is
c = -0.5

In [ ]:
size = 512
mask = np.zeros((size, size))
half = size // 2

# creating a boundary
mask[:half, half:] = c
mask[half:, :half] = c

In [ ]:
boundary_mask_vector = upper_triangular_to_vector_skip_two_diagonals(mask)
boundary_mask_vector_tensor = torch.tensor(boundary_mask_vector).to(device)

In [ ]:
# Create a blank mask
fragment_bool_mask = np.zeros((size, size), dtype=bool)

# Mark the specified fragment
fragment_bool_mask[:half, half:] = True
fragment_bool_mask[half:, :half] = True

In [ ]:
boundary_mask_indices = fragment_indices_in_upper_triangular(matrix_size=size, fragment_mask=fragment_bool_mask)
boundary_mask_indices_tensor = torch.tensor(boundary_mask_indices)

In [ ]:

# torch.save(boundary_mask_indices_tensor, "/scratch1/smaruj/generate_cell_type_specific_features/boundary_indices.pt")

In [ ]:
for row in df.itertuples(index=False):
    chrom, pred_start, pred_end = row.chrom, row.centered_start, row.centered_end
    sequence = genome[row.chrom][pred_start:pred_end]
    
    X = one_hot_encode_sequence(sequence)
    X_tensor = torch.tensor(X)
    # torch.save(X_tensor, f"/scratch1/smaruj/generate_genomic_boundary/ohe_X/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_X.pt")

    # store_tower_output(X_tensor, model, f"/scratch1/smaruj/generate_cell_type_specific_features/tower_output_mESC/model{model_idx}/{chrom}_{pred_start}_{pred_end}_tower_out.pt")
    # store_tower_output(X_tensor, model, f"/scratch1/smaruj/generate_cell_type_specific_features/tower_output_ncx_NPC/model{model_idx}/{chrom}_{pred_start}_{pred_end}_tower_out.pt")
    
    model.eval()
    with torch.no_grad():
        y = model(X_tensor)
        
    # replacing values of map with the mask
    y_bar = y.to(device).clone()
    # masked_values = boundary_mask_vector_tensor[boundary_mask_indices_tensor].float()
    # y_bar[0, 0, boundary_mask_indices_tensor] = masked_values.to(device)
    
    torch.save(y_bar, f"/scratch1/smaruj/generate_cell_type_specific_features/target_ncx_NPC_no_boundary/model{model_idx}/{chrom}_{pred_start}_{pred_end}_target.pt")